# fMRI Connectomics and Machine Learning

This notebook builds an fMRI connectomics pipeline from regional BOLD time series to functional connectivity, graph measures, and age-group classification. The analysis uses the Schaefer 2018 atlas with 100 regions and a subset of the Development fMRI dataset containing children and adults.

The main steps are regional signal extraction, connectivity estimation, graph analysis, leakage-aware classification, and interpretation of the linear SVM weights. A standard Python runtime is sufficient for the analysis.


In [ ]:
# Install Nilearn
!pip install nilearn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from networkx.algorithms import community

from nilearn import datasets, plotting
from nilearn.maskers import NiftiLabelsMasker
from nilearn.connectome import ConnectivityMeasure

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, average_precision_score, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif

## Brain atlas and development fMRI data

The brain is parcellated with the **Schaefer 2018 atlas** using 100 regions assigned to the 7-network configuration. These regions define the nodes used throughout the connectivity analysis.

The functional data come from 50 subjects in Nilearn's **Development fMRI dataset**, with the target defined as child versus adult. The atlas is first displayed on a standard anatomical background, then the functional images, confounds, and phenotypic labels are loaded for the subsequent analysis.


In [ ]:
print("Fetching Schaefer Atlas...")
atlas = datasets.fetch_atlas_schaefer_2018(n_rois=100, yeo_networks=7, resolution_mm=2)

print("Plotting Atlas...")
plotting.plot_roi(atlas.maps, title="Schaefer 2018 Atlas (100 ROIs)", cmap='nipy_spectral')
plt.show()

print("Fetching Development dataset (50 subjects)...")
dev_data = datasets.fetch_development_fmri(n_subjects=50)
fmri_filenames = dev_data.func
confounds = dev_data.confounds

# Encode the phenotypic child/adult labels
phenotypic = dev_data.phenotypic
labels = np.array([1 if x == 'child' else 0 for x in phenotypic['Child_Adult']])

print(f"Loaded {len(fmri_filenames)} subjects. Classes: {np.bincount(labels)} (0=Adult, 1=Child)")

## Regional time-series extraction

For each subject, `NiftiLabelsMasker` is used to extract the mean BOLD signal within each of the 100 atlas regions. The confound variables supplied with the dataset are included during signal cleaning, and the regional signals are standardized within each subject using sample-wise z-scoring.

Each subject is processed independently, producing a time-by-region matrix that is later used to estimate functional connectivity.


In [ ]:
masker = NiftiLabelsMasker(labels_img=atlas.maps, standardize='zscore_sample',
                           memory='nilearn_cache', verbose=0)

time_series_all = []
print("Extracting time-series. This will take a few minutes...")
for i, (func_img, confound_file) in enumerate(zip(fmri_filenames, confounds)):
    if (i+1) % 25 == 0:
        print(f"Processed {i+1}/{len(fmri_filenames)} subjects...")
    time_series = masker.fit_transform(func_img, confounds=confound_file)
    time_series_all.append(time_series)

print(f"Done! Shape of first subject's time-series: {time_series_all[0].shape} (Timepoints x ROIs)")

## Functional connectivity

Functional connectivity is estimated from the regional time series using both **correlation** and **partial correlation**. The two matrices for the first subject are displayed side by side to show how direct and marginal statistical relationships differ.

A connectome view is then produced from the correlation matrix by retaining the strongest 5% of edges, which gives a sparse representation of the dominant functional connections for that subject.


In [ ]:
# Correlation connectivity
conn_corr = ConnectivityMeasure(kind='correlation', standardize='zscore_sample')
matrices_corr = conn_corr.fit_transform(time_series_all)

# Partial-correlation connectivity
conn_part = ConnectivityMeasure(kind='partial correlation')
matrices_part = conn_part.fit_transform(time_series_all)

# Atlas coordinates used for connectome plots
coords = plotting.find_parcellation_cut_coords(labels_img=atlas.maps)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plotting.plot_matrix(matrices_corr[0], axes=axes[0], colorbar=True, vmax=1, vmin=-1, title="Subject 0 - Correlation")
plotting.plot_matrix(matrices_part[0], axes=axes[1], colorbar=True, vmax=1, vmin=-1, title="Subject 0 - Partial Corr")
plt.show()

print("Connectome with 10% proportional threshold:")
plotting.plot_connectome(matrices_corr[0], coords, edge_threshold="95%",
                         title="Subject 0 Connectome (Top 5% Edges)")
plotting.show()

## Graph-based description of the connectome

The correlation matrix of the first subject is converted into a binary graph after removing self-connections and retaining edges with an absolute correlation greater than 0.4. In this graph, atlas regions are nodes and retained functional connections are edges.

The resulting network is summarized using the average clustering coefficient, the highest-degree nodes, betweenness centrality, and modularity obtained from greedy community detection. These measures provide complementary descriptions of local clustering, hubs, potential bottlenecks, and community structure.


In [ ]:
# Absolute thresholding
subj_0_corr = matrices_corr[0].copy()
np.fill_diagonal(subj_0_corr, 0)  # Remove self-connections
adj_matrix = (np.abs(subj_0_corr) > 0.4).astype(int)

# Build the NetworkX graph
G = nx.from_numpy_array(adj_matrix)

# Graph summary measures
clustering = nx.average_clustering(G)

degrees = dict(G.degree())
top_degree = sorted(degrees.items(), key=lambda x: x[1], reverse=True)[:3]

betweenness = nx.betweenness_centrality(G)
top_betweenness = sorted(betweenness.items(), key=lambda x: x[1], reverse=True)[:3]

# Community structure and modularity
communities = community.greedy_modularity_communities(G)
modularity = community.modularity(G, communities)

print(f"Average Clustering Coefficient: {clustering:.3f}")
print(f"Top 3 Hubs by Degree (Node, Degree): {top_degree}")
print(f"Top 3 Bottlenecks by Betweenness: {[node for node, val in top_betweenness]}")
print(f"Graph Modularity: {modularity:.3f} (Number of communities: {len(communities)})")

# Graph visualization
plt.figure(figsize=(6,6))
nx.draw_spring(G, node_size=30, node_color='red', edge_color='gray', alpha=0.5)
plt.title("Subject 0 Thresholded Graph")
plt.show()

## Age-group classification and data leakage

The classification task distinguishes children from adults with a linear SVM and 5-fold stratified cross-validation. Tangent-space connectivity and univariate feature selection are used before classification.

Two evaluation strategies are compared. In the first, tangent connectivity and feature selection are fitted once on the complete dataset before cross-validation. This allows information from held-out subjects to influence preprocessing. In the second, `ConnectivityMeasure`, scaling, feature selection, and the SVM are placed inside a single scikit-learn pipeline, so every data-dependent step is fitted only on the training portion of each fold.

Performance is summarized with AUROC and AUPRC for both strategies.


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
model = SVC(probability=True, kernel='linear', random_state=42)
y = labels

# Tangent reference estimated once from the complete dataset
conn_leaky = ConnectivityMeasure(kind='tangent', vectorize=True)
X_leaky = conn_leaky.fit_transform(time_series_all)

# Feature selection also uses the complete set of labels
selector_leaky = SelectKBest(f_classif, k=100)
X_leaky_selected = selector_leaky.fit_transform(X_leaky, y)

# Cross-validation after global preprocessing
leaky_roc = cross_val_score(model, X_leaky_selected, y, cv=cv, scoring='roc_auc')
leaky_prc = cross_val_score(model, X_leaky_selected, y, cv=cv, scoring='average_precision')

# Leakage-controlled pipeline: all fitted preprocessing steps remain inside cross-validation
strict_pipe = Pipeline([
    ('conn', ConnectivityMeasure(kind='tangent', vectorize=True)),
    ('scaler', StandardScaler()),
    ('selector', SelectKBest(f_classif, k=100)),
    ('svm', model)
])

# Cross-validation starts from the subject-level time series
strict_roc = cross_val_score(strict_pipe, time_series_all, y, cv=cv, scoring='roc_auc')
strict_prc = cross_val_score(strict_pipe, time_series_all, y, cv=cv, scoring='average_precision')

# Approximate 95% interval across folds
def calc_ci(scores):
    mean = np.mean(scores)
    ci = 1.96 * np.std(scores) / np.sqrt(len(scores))
    return mean, ci

roc_leaky_m, roc_leaky_ci = calc_ci(leaky_roc)
roc_strict_m, roc_strict_ci = calc_ci(strict_roc)
prc_leaky_m, prc_leaky_ci = calc_ci(leaky_prc)
prc_strict_m, prc_strict_ci = calc_ci(strict_prc)

print("--- AUROC (mean ± approximate 95% interval) ---")
print(f"Global preprocessing: {roc_leaky_m:.3f} ± {roc_leaky_ci:.3f}")
print(f"Fold-wise pipeline:    {roc_strict_m:.3f} ± {roc_strict_ci:.3f}")

print("\n--- AUPRC (mean ± approximate 95% interval) ---")
print(f"Global preprocessing: {prc_leaky_m:.3f} ± {prc_leaky_ci:.3f}")
print(f"Fold-wise pipeline:    {prc_strict_m:.3f} ± {prc_strict_ci:.3f}")

## Interpreting the linear SVM

After cross-validation, the leakage-controlled pipeline is fitted to the complete dataset for model inspection. The SVM coefficients correspond only to the features retained by `SelectKBest`, so they are first mapped back to the full tangent-connectivity feature space.

The resulting weights are reconstructed as a connectivity matrix and the strongest 1% of discriminative connections are displayed on the brain. This visualization is used for interpretation of the fitted model rather than for estimating predictive performance.


In [ ]:
# Fit the final pipeline to the complete dataset for interpretation
strict_pipe.fit(time_series_all, y)

# Fitted pipeline components
conn_step = strict_pipe.named_steps["conn"]
selector_step = strict_pipe.named_steps["selector"]
svm_step = strict_pipe.named_steps["svm"]

# Linear SVM coefficients for the selected features
svm_weights = svm_step.coef_[0]

# Selected positions in the original connectivity vector
selected_indices = selector_step.get_support(indices=True)

# Number of connectivity features before selection
n_features = selector_step.n_features_in_

# Restore coefficients to the complete connectivity vector
full_weights = np.zeros(n_features)
full_weights[selected_indices] = svm_weights

print("Number of connectivity features:", n_features)
print("Number selected:", len(selected_indices))
print("Number of SVM weights:", len(svm_weights))

# Reconstruct the connectivity-weight matrix
weight_matrix = conn_step.inverse_transform(
    full_weights.reshape(1, -1)
)[0]

print("Visualising the most discriminative edges (Child vs Adult):")

plotting.plot_connectome(
    weight_matrix,
    coords,
    edge_threshold="99%",
    title="Top 1% SVM Weights (Feature Importance)",
    colorbar=True,
    node_size=20,
)

plotting.show()

## Additional comparisons

The same pipeline can be used to examine how modelling choices affect predictive performance and interpretation. Relevant comparisons include replacing the linear SVM with a `RandomForestClassifier`, increasing the number of retained features from 100 to 500 or all available features, and comparing tangent-space connectivity with conventional correlation features.

For tree-based models, `feature_importances_` can be examined in place of the linear SVM coefficients, while the same cross-validation structure should be retained so that preprocessing remains confined to the training folds.
